# Stage 3 — Downstream: does pretraining pay off?

Runs, in one notebook:
1. **Feature caching** for all four arms (our MAE-single, our MAE-cross,
   ImageNet-pretrained ViT-S, scratch ViT-S) — needs Internet ON once for timm.
2. **Linear probes** with subject-level GroupKFold + label-efficiency fractions
   (10/25/50/100 % of training *subjects*, same subset for every arm).
3. **Fine-tuning** (attention-pool head) for the two MAE arms.
4. **Age probe** sanity check on frozen features.

All results land in `results/` as CSVs — every table in the README comes from
them. Budget: feature cache ~20–40 min; probes minutes; finetune ~1–3 h.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_KAGGLE = Path("/kaggle").is_dir()
if IN_KAGGLE:
    repo = Path("mv-brainmae")
    if not repo.is_dir():
        # set your repo URL after pushing to GitHub
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/YOUR_GITHUB_USERNAME/mv-brainmae.git"],
                       check=True)
else:
    repo = Path("..")  # local: notebooks/ inside a clone
sys.path.insert(0, str(repo.resolve()))
os.chdir(repo)
print("repo:", repo.resolve())

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from mvbrainmae.utils import load_config
from mvbrainmae.downstream import run_downstream

cfg = load_config("configs/downstream.yaml")
summary = run_downstream(cfg)

In [ ]:
# ---- VERIFICATION ----
import pandas as pd
probe = pd.read_csv("results/downstream_probe.csv")
print(probe.pivot_table(index=["encoder", "label_frac"], columns="scope",
                        values="balanced_accuracy").round(3))
ft = pd.read_csv("results/downstream_finetune.csv")
print(ft.groupby("encoder")["balanced_accuracy"].agg(["mean", "std"]).round(3))
age = pd.read_csv("results/downstream_age.csv")
print(age[["encoder", "pearson_r", "mae_years"]].round(3))
# leakage audit: test subjects must never appear in any training predictions file
sp = pd.read_csv("data/factory/splits.csv").set_index("subject_id").split
for f in sorted(Path("results").glob("preds_probe_*frac1.0.csv")):
    d = pd.read_csv(f)
    assert set(sp[d.subject].unique()) == {"test"}, f"LEAK in {f}"
print("leakage audit OK: prediction files contain TEST subjects only")